# Nemotron Reasoning Challenge — training notebook

Clean pipeline matching `scripts/` on branch `build/nemotron-pipeline`.

**Before running:** Add Input → (1) the **competition data**, (2) the model **`nemotron-3-nano-30b-a3b-bf16`** (publisher `metric`). Internet **ON** is the simplest setup. See `WRITEUP.md` for the method and `kaggle_wheels/README.md` for the internet-OFF path.

## 1. Get the code

In [ ]:
!rm -rf repo && git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!ls scripts

## 2. Dependencies (internet ON) + availability check
The competition's Nemotron image usually already ships `mamba_ssm` + `causal_conv1d`. If they show **MISS**, install matching wheels with `kaggle_wheels/fetch_torch_locked_wheels.py` (see that folder's README).

In [ ]:
!pip install -q -U peft trl datasets accelerate bitsandbytes
import importlib
for m in ["torch","transformers","peft","trl","datasets","accelerate","bitsandbytes","mamba_ssm","causal_conv1d"]:
    try: importlib.import_module(m); print('ok  ', m)
    except Exception as e: print('MISS', m, '->', repr(e)[:80])

## 3. Bring in the competition data

In [ ]:
import glob, os, shutil
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/*reasoning*/train.csv') + glob.glob('/kaggle/input/*nemotron*/train.csv')
assert hits, "train.csv not found — Add Input -> the competition dataset"
shutil.copy(hits[0], 'data/train.csv'); print('train.csv <-', hits[0])

## 4. EDA + build the SFT data (local, fast, no GPU)

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data   # -> data/train_sft.jsonl

## 5. Train the LoRA adapter (2xT4)
8-bit + CPU offload. A smoke test runs first to prove the memory config fits. If it OOMs, the script writes **`FALLBACK.md`** — run this one step on a rented A100/H100 with the same config and upload the resulting `lora_adapter/` as a Kaggle dataset instead.

In [ ]:
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir /kaggle/working/lora_adapter

## 6. (Optional) Local eval — needs vLLM
Mirrors the metric on a 10% per-category holdout. Skip if vLLM isn't available.

In [ ]:
!python scripts/04_evaluate.py --adapter-path /kaggle/working/lora_adapter --data-dir data

## 7. Package the submission
Writes `submission.zip` with the adapter files at the zip root (r<=32 checked).

In [ ]:
!python scripts/05_package_submission.py --adapter-dir /kaggle/working/lora_adapter --output /kaggle/working/submission.zip